# 远程仓库协作

前面的课程都在本地操作。本节将 Git 连接到远程仓库，实现多人协作。

你将学到：
1. 远程仓库的概念
2. `git clone`：克隆远程仓库
3. `git remote`：管理远程地址
4. `git push` / `git pull` / `git fetch`：推送与拉取
5. 跟踪分支与上游关联
6. 常见协作场景


---

## 1. 远程仓库的概念

远程仓库（remote repository）是托管在服务器上的版本库，用于团队协作的“中央交换台”。

常见的远程托管平台：

| 平台 | 特点 |
|------|------|
| **GitCode** | 国内平台，访问快，支持 Notebook 在线运行，本课程所在平台 |
| GitHub | 全球最大的代码托管平台 |
| Gitee | 国内代码托管平台 |
| GitLab | 可自建的开源平台 |

> Git 是分布式的——你可以同时关联多个远程仓库，这在 Fork 工作流中非常常见（origin = 你的 fork，upstream = 原仓库）。

---

## 2. git clone：克隆远程仓库

`git clone` 把远程仓库完整下载到本地，包括所有分支和完整历史。

### 2.1 HTTPS 方式（常用）

```bash
git clone https://gitcode.com/cann/cann-learning-hub.git
```

- ✅ 防火墙友好，几乎任何网络都能用
- ❌ 每次 push 需要输入用户名和令牌（可配置凭证缓存）

### 2.2 SSH 方式

```bash
git clone git@gitcode.com:cann/cann-learning-hub.git
```

- ✅ 配置好 SSH 密钥后免密操作
- ✅ 更安全
- ❌ 需要预先配置 SSH 密钥


In [ ]:
# 克隆课程仓（HTTPS 方式，见 2.1 节），完整历史
import os, shutil
repo = "/tmp/cann-learning-hub"
os.chdir("/tmp")   # 避免待在 repo 里删自己
if os.path.exists(repo):
    shutil.rmtree(repo)

!git clone https://gitcode.com/cann/cann-learning-hub.git /tmp/cann-learning-hub


In [ ]:
# 查看本课程仓库的远程信息（我们已经 clone 过了）
import os, subprocess
repo = "/tmp/cann-learning-hub"
if not os.path.isdir(repo):
    # 没跑过 2.1 节的克隆 cell 也不要紧：自动补一次克隆
    print("仓库不存在，自动克隆一次……（约 250MB，需几分钟）")
    subprocess.run(["git", "clone", "https://gitcode.com/cann/cann-learning-hub.git", repo], check=True)
os.chdir(repo)

!git remote -v

# 查看远程分支
!git branch -r


---

## 3. git remote：管理远程地址

### 3.1 查看远程

```bash
git remote              # 列出所有远程名
git remote -v           # 显示远程名 + URL（push 和 fetch）
```

克隆仓库时，Git 自动把远程命名为 **origin**。

### 3.2 添加远程

在 Fork 工作流中，你通常需要两个远程：

| 远程名 | 指向 | 权限 |
|--------|------|------|
| `origin` | 你 fork 的仓库 | 可读写（你的） |
| `upstream` | 原始仓库 | 只读（别人的） |

```bash
# 添加 upstream 远程
git remote add upstream https://gitcode.com/cann/cann-learning-hub.git

# 验证
git remote -v
```

### 3.3 修改和删除远程

```bash
# 修改远程 URL（例如从 HTTPS 切换到 SSH）
git remote set-url origin git@gitcode.com:cann/cann-learning-hub.git

# 删除远程
git remote remove upstream

# 重命名
git remote rename origin myfork
```

In [ ]:
# 动手练习：远程的添加、修改、重命名、删除（在刚才克隆的练习仓库上，随便折腾）
import os, subprocess
repo = "/tmp/cann-learning-hub"
if not os.path.isdir(repo):
    # 没跑过 2.2 节的克隆 cell 也不要紧：自动补一次克隆（约 250MB，需几分钟）
    print("练习仓库不存在，自动克隆一次……")
    subprocess.run(["git", "clone", "--depth", "1",
                    "git@gitcode.com:cann/cann-learning-hub.git", repo], check=True)
os.chdir(repo)

# 添加远程（3.2 节）——Fork 流程中就是这样添加 upstream 的
!git remote add upstream https://gitcode.com/cann/cann-learning-hub.git
!git remote -v

# 修改 URL（3.3 节）——例如把 HTTPS 换成 SSH
!git remote set-url upstream git@gitcode.com:cann/cann-learning-hub.git
!git remote -v

# 重命名远程
!git remote rename upstream myfork

# 删除远程
!git remote remove myfork

# 折腾完只剩 origin
!git remote -v


---

## 4. git push / git pull / git fetch

### 4.1 git push：推送本地提交到远程

```bash
git push origin master              # 推送 main 分支到 origin
git push origin feature/login     # 推送指定分支
git push origin                   # 推送当前分支（已设置上游时）
git push -u origin feature/login  # 首次推送并设置上游（-u = --set-upstream）
git push --force-with-lease       # 安全的强制推送（推荐，比 --force 更安全）
```

> ⚠️ 永远避免对公共分支（master/main 等）使用 `git push --force`，这会覆盖他人的提交！

### 4.2 git fetch：下载但不合并

`git fetch` 只下载远程的更新到本地，**不修改**你的工作区。你可以先 fetch 看看远程发生了什么，再决定是否合并。

```bash
git fetch origin              # 下载 origin 的所有分支更新
git fetch upstream master     # 只下载 upstream 的 master 分支

# fetch 后查看远程分支的变化
git log --oneline origin/master..HEAD   # 本地领先远程的提交
git log --oneline HEAD..origin/master   # 远程领先本地的提交
```

### 4.3 git pull：下载并合并

`git pull` = `git fetch` + `git merge`。它下载远程更新并直接合并到当前分支。

```bash
git pull origin master          # 拉取并合并 origin/master
git pull                      # 拉取并合并当前分支的上游
git pull --rebase             # 拉取后用 rebase 代替 merge（保持线性历史）
```

> 💡 **fetch + merge vs pull**：`git pull` 简单但有风险（可能产生意外的合并提交）。推荐新手先 `git fetch` 再 `git merge`，清楚知道每一步在做什么。

---

## 5. 跟踪分支与上游关联

跟踪分支（tracking branch）让本地分支“记住”它对应的远程分支，之后 `git push` / `git pull` 不用再指定远程和分支。

```bash
# 方式 1：首次推送时设置上游
git push -u origin feature/login

# 方式 2：为已有分支设置上游
git branch -u origin/feature/login

# 方式 3：基于远程分支创建本地跟踪分支
git checkout -b feature/login origin/feature/login
# 简写（Git 2.x+ 自动跟踪）
git checkout feature/login
```

设置后，`git status` 会显示本地与远程的领先/落后关系：

```
On branch feature/login
Your branch is up to date with 'origin/feature/login'.
Your branch is ahead of 'origin/feature/login' by 2 commits.
```

---

## 6. 常见协作场景

### 6.1 同步上游仓库的更新

当你 Fork 了别人的仓库，原仓库（upstream）可能有新提交。同步方法：

```bash
# 1. 拉取 upstream 的最新代码
git fetch upstream

# 2. 切到 master 分支
git checkout master

# 3. 合并 upstream 的 master
git merge upstream/master

# 4. 推送到你的 fork
git push origin master
```

### 6.2 获取同事的分支

同事推送了一个新分支 `feature/payment`，你想在本地查看：

```bash
git fetch origin
git checkout feature/payment    # 自动创建跟踪分支
```

### 6.3 误推了不该推的文件

```bash
# 从远程分支删除文件，但保留本地
git rm --cached sensitive.txt
git commit -m "fix: 从仓库移除敏感文件"
git push origin master

# 别忘了把它加入 .gitignore！
echo "sensitive.txt" >> .gitignore
```

---

## 7. 总结

**回顾远程协作核心命令：**

```bash
git clone <url>                # 克隆仓库
git remote -v                  # 查看远程
git remote add upstream <url>  # 添加上游远程
git fetch <remote>             # 下载更新（不合并）
git pull <remote> <branch>     # 下载并合并
git push <remote> <branch>     # 推送提交
git push -u origin <branch>    # 首次推送并设置上游
```
接下来，请学习 [Git 指令汇总](./05_git_order.ipynb)，随时查阅常用指令！
